# Extract 2AFC data from the csv file generated by jsPsych

Use this notebook for the 2AFC tasks: object recognition and drawing recognition. It reads raw jsPsych CSV files and makes a cleaner table with image number, response, correct answer, and accuracy.

## What can I change?

At the bottom of this notebook, change the input folder or file name so it points to your own data.


In [18]:
import os
import re
import glob
import pandas as pd
import numpy as np
import json

In [19]:
def extract_image_idx(stimulus):
    if pd.isna(stimulus):
        return np.nan

    fname = os.path.basename(str(stimulus))
    match = re.search(r"(\d+)", fname)

    if match:
        return int(match.group(1))

    return np.nan


def parse_survey_response(response):
    if pd.isna(response):
        return {}

    try:
        return json.loads(response)
    except:
        return {}


def age_to_midpoint(age_str):
    if pd.isna(age_str):
        return np.nan

    age_str = str(age_str)

    if "-" in age_str:
        lo, hi = age_str.split("-")
        return (float(lo) + float(hi)) / 2

    if "+" in age_str:
        return float(age_str.replace("+", ""))

    return float(age_str)


def extract_2afc_file(csv_file):
    subject_id = os.path.splitext(os.path.basename(csv_file))[0].split("_")[-1]
    df = pd.read_csv(csv_file)

    # Mark stimulus rows
    df["stimulus_image"] = df["stimulus"].where(
        df["trial_type"] == "image-button-response"
    )

    df["image_idx"] = df["stimulus_image"].apply(extract_image_idx)

    # Carry stimulus forward to the later response row
    df["stimulus_image"] = df["stimulus_image"].ffill()
    df["image_idx"] = df["image_idx"].ffill()


    # Object-recognition responses are html-button-response rows
    trials = df[
        (df["trial_type"] == "html-button-response")
        & df["correct"].notna()
        & df["response"].notna()
    ].copy()

    trials["response"] = pd.to_numeric(trials["response"], errors="coerce")
    trials["correct_choice"] = pd.to_numeric(trials["correct_choice"], errors="coerce", downcast='integer')
    trials["correct"] = trials["correct"].astype(bool)
    trials["image_idx"] = pd.to_numeric(trials["image_idx"], errors="coerce", downcast='integer')

    trials["source_file"] = os.path.basename(csv_file)
    trials["subject_id"] = subject_id

    out = trials[
        [
            "source_file",
            "subject_id",
            "rt",
            "stimulus_image",
            "image_idx",
            "response",
            "correct_choice",
            "choices",
            "correct",
        ]
    ].copy()

    return out

In [22]:
def extract_2afc_folder(input_folder, output_csv="obj_2afc.csv"):
    all_files = glob.glob(os.path.join(input_folder, "*.csv"))

    all_data = []
    ages = []
    sexes = []

    excluded = 0
    for f in all_files:
        if "extracted" in f:
          excluded += 1
          continue
        raw_df = pd.read_csv(f)

        # Survey row
        survey_rows = raw_df[raw_df["trial_type"] == "survey-multi-choice"]

        if len(survey_rows) > 0:
            survey_response = survey_rows.iloc[0]["response"]
            survey = parse_survey_response(survey_response)

            if "Age" in survey:
                ages.append(survey["Age"])

            if "Gender" in survey:
                sexes.append(survey["Gender"])

        parsed = extract_2afc_file(f)
        all_data.append(parsed)

    results = pd.concat(all_data, ignore_index=True)
    results.to_csv(output_csv, index=False)

    # ---------------------------------
    # Print survey statistics
    # ---------------------------------
    print("\n==============================")
    print("Survey Statistics")

    print(f"N participants: {len(all_files)-excluded}")

    if len(ages) > 0:
        age_counts = pd.Series(ages).value_counts()
        print("\nAge distribution:")
        for age, count in age_counts.items():
            print(f"  {age}: {count}")

    if len(sexes) > 0:
        sex_counts = pd.Series(sexes).value_counts()

        print("\nGender distribution:")
        for sex, count in sex_counts.items():
            print(f"  {sex}: {count}")

    print("\n==============================")
    print("2AFC Performance")

    print(f"N trials: {len(results)}")
    print(f"Mean accuracy: {results['correct'].mean():.3f}")
    print(f"Mean RT: {results['rt'].mean():.2f} ms")

    print("\nSaved data to:")
    print(output_csv)

    return results

In [ ]:
# ===============================
# STUDENTS: EDIT THIS PART BELOW
# ===============================

In [ ]:
# Example: parse one file: object 2AFC file
results = extract_2afc_file("./data/obj_2AFC/obj_2afc_484959.csv")
results.to_csv("./data/obj_2AFC/obj_2afc_484959_extracted.csv", index=False)

In [ ]:
# Example: parse one drawing 2AFC file
results = extract_2afc_file("./data/draw_2AFC/draw_2afc_963156.csv")
results.to_csv("./data/draw_2AFC/draw_2afc_963156_extracted.csv", index=False)

In [23]:
# Example: parse all files in one folder
results = extract_2afc_folder("./data/obj_2AFC", "./data/all_data_obj_2afc.csv")


Survey Statistics
N participants: 1

Age distribution:
  26-30: 1

Gender distribution:
  Female: 1

2AFC Performance
N trials: 5
Mean accuracy: 0.600
Mean RT: 1301.42 ms

Saved data to:
./data/all_data_obj_2afc.csv


In [24]:
# Example: parse all drawing-recognition files in one folder
results = extract_2afc_folder("./data/draw_2AFC", "./data/all_data_draw_2afc.csv")


Survey Statistics
N participants: 1

Age distribution:
  26-30: 1

Gender distribution:
  Female: 1

2AFC Performance
N trials: 4
Mean accuracy: 0.750
Mean RT: 671.92 ms

Saved data to:
./data/all_data_draw_2afc.csv
